# Tutorial 2a: metafspm writes and reads encoding3d

[metafspm](https://github.com/openalea/metafspm) is OpenAlea's framework for plant models made of **components**: one
for root growth, one for water flow, one for light... A **scene** grows several plants together. This tutorial shows
how a metafspm scene is saved as encoding3d files, and read back:

1. a small root growth component, written here for the tutorial (a toy, not a real model);
2. a scene of two plants, a frame recorded at each step;
3. those frames read with encoding3d alone, as any other platform would;
4. smaller frames, keeping only some scales or variables;
5. a frame turned back into metafspm's data structures, exactly as they were.

**What you need:** metafspm and encoding3d installed in the same environment (this folder's `README.md`).

In [ ]:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

from openalea import encoding3d
from openalea.encoding3d.validate import validate_sequence
from openalea.metafspm.coupling.component import StructuralComponent, parameter, state_variable
from openalea.metafspm.data_structure.configs import PropsConfig, ScalesConfig as scales
from openalea.metafspm.io.encoding3d import import_frame
from openalea.metafspm.scene.scene import Scene, SceneRecorder
from openalea.metafspm.solve.decorator import actual, segmentation

OUTPUTS = "outputs"

## 1. A root growth component

A metafspm component declares its **variables** (with units, the scale they live at, their kind) and its
**processes**, the methods run at each time step. Ours grows one root axis: its tip elongates at a constant rate,
and when it is longer than 1 cm it is cut, a new tip continuing it. Each segment knows its two ends (`x1` … `z2`) and
its radius, so it can be drawn.

`initiate_plant` builds each plant's first segment, under the scales metafspm uses (plant, axis, growth unit,
phytomer, organ, segment).

In [ ]:
SEGMENT = 0.01                                           # m
UNIT = dict(unit_comment="", value_comment="", references="", DOI=[], min_value=None, max_value=None)


@dataclass
class ToyRootGrowth(StructuralComponent):
    """A root axis: the tip elongates at a constant rate, cut every 1 cm (a toy, for the tutorial)."""
    length: float = state_variable(unit="m", description="the segment's length", default=0., scale=scales.SubOrgan,
                                   kind="extensive", **UNIT)
    radius: float = state_variable(unit="m", description="the segment's radius", default=0.0005,
                                   scale=scales.SubOrgan, kind="intensive", **UNIT)
    is_apex: float = state_variable(unit="adim", description="1 for the tip", default=0., scale=scales.SubOrgan,
                                    kind="descriptor", **UNIT)
    x1: float = state_variable(unit="m", description="proximal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    y1: float = state_variable(unit="m", description="proximal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    z1: float = state_variable(unit="m", description="proximal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    x2: float = state_variable(unit="m", description="distal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    y2: float = state_variable(unit="m", description="distal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    z2: float = state_variable(unit="m", description="distal end", default=0., scale=scales.SubOrgan,
                               kind="descriptor", **UNIT)
    elongation_rate: float = parameter(unit="m.s-1", description="the tip's elongation rate", default=2e-7,
                                       **UNIT)

    @classmethod
    def initiate_plant(cls, g, plant, scenario):
        s = g.scales
        axis = g.add_component(plant, **PropsConfig(scale=s.Axis, edge_type="/", label=g.labels.Axis.Root))
        unit = g.add_component(axis, **PropsConfig(scale=s.GrowthUnit, edge_type="/", label=g.labels.GrowthUnit.Root))
        phytomer = g.add_component(unit, **PropsConfig(scale=s.Phytomer, edge_type="/", label=g.labels.Phytomer.Root))
        organ = g.add_component(phytomer, **PropsConfig(scale=s.Organ, edge_type="/",
                                                        label=g.labels.Organ.RootInternode))
        tip = g.add_component(organ, **PropsConfig(scale=s.SubOrgan, edge_type="/", label=g.labels.SubOrgan.RootSegment))
        x, y, z = (g.property(name)[plant] for name in ("x", "y", "z"))
        for name, value in (("length", 0.002), ("radius", 0.0005), ("is_apex", 1.), ("x1", x), ("y1", y), ("z1", z),
                            ("x2", x), ("y2", y), ("z2", z - 0.002)):
            g.property(name)[tip] = value

    def _tips(self):
        is_apex = self.mtg.property("is_apex")
        return [v for v, flag in is_apex.items() if flag]

    @actual
    def _elongation(self):
        """The tips grow downwards."""
        g = self.mtg
        length, z2 = g.property("length"), g.property("z2")
        rate = dict(zip(self.data_structure.entity_ids("node").tolist(),
                        self.parameter_values("elongation_rate").tolist()))
        for tip in self._tips():
            growth = rate[tip] * self.dt
            length[tip] += growth
            z2[tip] -= growth

    @segmentation
    def _segmentation(self):
        """A tip longer than 1 cm becomes a segment, continued by a new tip."""
        g = self.mtg
        props = {name: g.property(name) for name in ("length", "is_apex", "radius", "x1", "y1", "z1", "x2", "y2", "z2")}
        for tip in self._tips():
            if props["length"][tip] <= SEGMENT:
                continue
            rest = props["length"][tip] - SEGMENT
            props["length"][tip], props["is_apex"][tip] = SEGMENT, 0.
            props["z2"][tip] = props["z1"][tip] - SEGMENT
            x, y, z = props["x2"][tip], props["y2"][tip], props["z2"][tip]
            g.add_child(tip, **PropsConfig(scale=g.scales.SubOrgan, edge_type="<", label=g.labels.SubOrgan.RootSegment,
                                           length=rest, is_apex=1., radius=props["radius"][tip],
                                           x1=x, y1=y, z1=z, x2=x, y2=y, z2=z - rest))


class ToyRoots:
    """The plant model: one component, run once per step."""
    initiators = (ToyRootGrowth,)

    def __init__(self, data_structure, time_step):
        self.growth = ToyRootGrowth(data_structure=data_structure)
        self.components = [self.growth]

    def run(self):
        self.growth()

## 2. A scene, recorded

The **planting table** says which plants grow where. `SceneRecorder(frame_period=1)` writes a frame at the start and
after every step, with the index `scene.fspm.json`: an encoding3d sequence.

In [ ]:
planting = pd.DataFrame([dict(plant=f"plant_{i}", model=ToyRoots, x=0.05 * i, y=0., z=0., rotation=0., scenario={})
                         for i in range(2)])
planting.attrs.update(xrange=0.2, yrange=0.1)
scene = Scene(planting=planting, time_step=6 * 3600., recorder=SceneRecorder(os.path.join(OUTPUTS, "full"),
                                                                             summaries=False, frame_period=1))
for _ in range(8):                                     # two days, a step every six hours
    scene.run()
index = os.path.join(OUTPUTS, "full", "frames", "scene.fspm.json")
report = validate_sequence(index)
# warnings: metafspm also writes its graph's own properties (positions, types), which have no declared unit
print("valid:", report.ok, "|", len(report.errors), "errors,", len(report.warnings), "warnings")

## 3. The frames, read by encoding3d alone

Nothing in these files needs metafspm: any platform reads them. Each MPG scale is an entity set, every variable a
column with its unit, and the segments' triangles a mesh.

In [ ]:
rows = []
for frame in encoding3d.read_sequence(index):
    plants = frame.data_structures[0]
    segments = plants.entity_set("SubOrgan")
    rows.append(dict(time_h=frame.time / 3600 if frame.time_unit == "s" else frame.time, step=frame.step,
                     segments=segments.count, total_length_cm=100 * segments.columns["length"].sum()))
print("entity sets:", [entity_set.name for entity_set in plants.entity_sets])
print("length's declaration:", segments.variables["length"])
pd.DataFrame(rows)

## 4. Smaller frames for logging

A long simulation writes many frames. `frame_options` keeps only what is needed: here the plants and the segments
(not the intermediate scales), and two variables. Such frames are valid encoding3d files, for outputs only.

In [ ]:
scene = Scene(planting=planting, time_step=6 * 3600.,
              recorder=SceneRecorder(os.path.join(OUTPUTS, "log"), summaries=False, frame_period=1,
                                     frame_options=dict(scales=["Plant", "SubOrgan"], variables=["length", "radius"],
                                                        geometry=False)))
for _ in range(8):
    scene.run()
log = os.path.join(OUTPUTS, "log", "frames")
full = os.path.join(OUTPUTS, "full", "frames")
size = lambda folder: sum(os.path.getsize(os.path.join(folder, name)) for name in os.listdir(folder))
small = encoding3d.read(os.path.join(log, sorted(os.listdir(log))[-2]))
print([entity_set.name for entity_set in small.data_structures[0].entity_sets],
      small.data_structures[0].entity_set("SubOrgan").variable_names())
print(f"full: {size(full) / 1e3:.0f} kB, log: {size(log) / 1e3:.0f} kB")

## 5. Back into metafspm

A full frame rebuilds metafspm's data structures exactly: the same graph, the same variables, bit for bit. A
simulation can continue from it.

In [ ]:
last = sorted(name for name in os.listdir(full) if name.endswith(".glb"))[-1]
rebuilt = import_frame(os.path.join(full, last))
population = scene.populations[0].name
ds = rebuilt[population]
print(population, "has", ds.n_nodes(), "segments; total length",
      round(100 * float(np.sum(ds.get("length"))), 3), "cm")

**In ParaView:** with the plugin loaded (tutorial 1), open `outputs/full/frames/scene.fspm.json` with the encoding3d
reader and press *Play*: the two roots grow.

## Try it yourself

- Change `elongation_rate` for one plant only, through its `scenario` in the planting table (`{"elongation_rate": 4e-7}`).
- Record a frame every four steps (`frame_period=4`) and compare the files.
- Keep the plants' scale only in the log, and read their number of segments from the full frames instead.